In [1]:
import sys,os
sys.path.append(os.path.join(os.environ['HOME'], 'models', 'OLMT'))
import numpy as np
import pickle
import model_ELM
import matplotlib.pyplot as plt
from string import ascii_lowercase
import pandas as pd
from scipy.stats import pearsonr
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")  # for inline plots
import matplotlib.path as mpath
import matplotlib.patches as mpatches


def draw_side_bracket(ax, y0, y1, *, x_ax=-0.12, cap_w_ax=0.03, side='left',
                      linewidth=1.2, zorder=5):
    """
    Draw a vertical bracket spanning [y0, y1] along the y-axis.
    x_ax:   x position in axes coords (e.g., -0.12 for left outside, 1.08 for right)
    cap_w_ax: horizontal width of the little end-caps (axes coords)
    side:   'left' (caps point right) or 'right' (caps point left)
    """
    yaxis_transform = ax.get_yaxis_transform()

    # Ensure y0 <= y1 (works fine even with inverted axis)
    y0, y1 = (y0, y1) if y0 <= y1 else (y1, y0)

    # Vertical spine of the bracket
    ax.plot([x_ax, x_ax], [y0, y1],
            transform=yaxis_transform, color='black',
            linewidth=linewidth, clip_on=False, zorder=zorder)

    # Cap direction: rightward for left side, leftward for right side
    cap_dx = +cap_w_ax if side == 'left' else -cap_w_ax

    # End caps
    ax.plot([x_ax, x_ax + cap_dx], [y0, y0],
            transform=yaxis_transform, color='black',
            linewidth=linewidth, clip_on=False, zorder=zorder)
    ax.plot([x_ax, x_ax + cap_dx], [y1, y1],
            transform=yaxis_transform, color='black',
            linewidth=linewidth, clip_on=False, zorder=zorder)

prefix_list = ['UQ_20231118', 'UQ_20240107', 'UQ_20240112']
name_list = ['ELM-OLD$_{optim}$_ENS', 'ELM-MYCI_ENS', 'ELM-MYCI$_{optim}$_ENS']
pft_names = ['Spruce','Tamarack','Shrub','Moss']

VAR_COL = ['GPP', 'NEE', 'HR', 'TOTVEGC', 'TOTSOMC']
VAR_PFT = ['GPP', 'AR', 'MR', 'GR', 'XR']
# variables for Xiaoying Shi
##VAR_COL = ['GPP', 'NPP', 'QVEGT', 'NEE', 'TOTVEGC']
##VAR_PFT = ['GPP', 'NPP', 'QVEGT']
pft_list = [2, 3, 11, 12]
nvars = len(VAR_COL) + len(pft_list) * len(VAR_PFT)


# mapping from the name in parameter file to paper notations
parmfile_to_paper = {
    'inh_fungi_a': 'a$_j$', 
    'inh_fungi_b': 'b$_j$',
    'q10_upt': 'q$_{10}$',
    'vmax_fungi_son': 'u$_{N,fungi,j}$',
    'vmax_fungi_sop': 'u$_{P,fungi,j}$',
    'km_froot_n': 'k$_{N,j}$',
    'km_froot_p': 'k$_{P,j}$',
    'vmax_fungi_din': 'v$_{N,fungi,j}$',
    'vmax_fungi_dip': 'v$_{P,fungi,j}$',
    'vmax_froot_n': 'v$_{N,froot,j}$',
    'vmax_froot_p': 'v$_{P,froot,j}$',
    'fungi_cost_n': 'c$_N$'
}


# break out the individual PFTs here by appending _{pft} to varname
variable_list = []
for var in VAR_COL:
    variable_list.append(var)
    if var in VAR_PFT:
        variable_list.extend([var+'_pft'+str(pft) for pft in pft_list])
for var in VAR_PFT:
    if not var in VAR_COL:
        variable_list.extend([var+'_pft'+str(pft) for pft in pft_list])


ticklabels = []
for var in VAR_COL:
    ticklabels.append(var)
    if var in VAR_PFT:
        # let PFT name be labeled separately horizontally 
        ticklabels.extend([var for pname in pft_names])
for var in VAR_PFT:
    if not var in VAR_COL:
        ticklabels.extend([var for pname in pft_names])

x_pos = np.cumsum(np.ones(len(variable_list)))

# reorder the variable list
reorder = np.array([0, 5, 6, 7, 8, 1, 9, 13, 17, 21, 2, 10, 14, 18, 22, 
                    3, 11, 15, 19, 23, 4, 12, 16, 20, 24])

In [2]:
for p, prefix in enumerate(prefix_list):
    f = open(os.path.join(os.environ['HOME'], 'models', 'OLMT', 'pklfiles', 
             f'{prefix}_US-SPR_ICB20TRCNPRDCTCBC.pkl'), 'rb')
    mycase = pickle.load(f)
    f.close()

    fig, axes = plt.subplots(2, 4, figsize = (16, 12), sharex = False, sharey = True)
    fig.subplots_adjust(wspace = 0.05, hspace = 0.18)

    for i, (pft,pftname) in enumerate(zip([2, 3, 11, 0], pft_names[:-1] + ['Column'])):
        subset = np.where(np.array(mycase.ensemble_pfts) == pft)[0]

        #Plot main/total sensitivity indices
        for j, effect in enumerate(['Main','Total']):
            ax = axes[j,i]

            if j == 0:
                ax.set_xlim([0., 1.1])
            elif j == 1:
                if p == 0:
                    ax.set_xlim([0., 1.41])
                elif p == 1:
                    ax.set_xlim([0., 1.95])
                elif p == 2:
                    ax.set_xlim([0., 2.4])

            bottom = np.zeros(len(x_pos))
            for s in subset:
                if effect == 'Main':
                    temp = np.array([mycase.sens_main[v][s,0] for v in variable_list])
                else:
                    temp = np.array([mycase.sens_tot[v][s,0] for v in variable_list])
                temp = temp[reorder]
                if p == 0:
                    ax.barh(x_pos, temp, align='center', # alpha=0.5,
                            left=bottom, label = mycase.ensemble_parms[s])
                else:
                    ax.barh(x_pos, temp, align='center', # alpha=0.5,
                            left=bottom, label = parmfile_to_paper[mycase.ensemble_parms[s]])
                bottom = bottom + temp

            # add a line for total
            if effect == 'Main':
                total = np.array([mycase.sens_main[v][:,0].sum() for v in variable_list])
            else:
                total = np.array([mycase.sens_tot[v][:,0].sum() for v in variable_list])
            total = total[reorder]
            if i == 3:
                ax.plot(total, x_pos, '-', color = 'grey', lw = 2, label = 'Sum of all perturbed parameters')
            else:
                ax.plot(total, x_pos, '-', color = 'grey', lw = 2, label = 'Sum of all\nperturbed\nparameters')

            ax.set_ylim([x_pos[0]-0.5, x_pos[-1]+0.5])
            ax.set_yticks(x_pos)
            ax.invert_yaxis()

            if i == 0:
                ax.set_yticklabels([ticklabels[t] for t in reorder])
                for pp, pname in enumerate(['Column'] + pft_names):
                    ax.text(-0.33, 0.9 - pp*0.2, pname, transform = ax.transAxes,
                            va = 'center', color = 'k', rotation = 90)

                # Use y-axis transform so x is in axes-fraction (fixed left/right of axis),
                # and y is in data coords (your tick positions).
                spans = [(1.5, 4.5), (6.5, 9.5), (11.5, 14.5), (16.5, 19.5), (21.5, 24.5)]

                # Draw on the LEFT of the axis
                for (y0, y1) in spans:
                    draw_side_bracket(ax, y0-0.4, y1+0.4, x_ax=-0.28, cap_w_ax=0.015, side='left')

            ax.set_title(f'{pftname} parameters')

            if j == 1:
                if p == 0:
                    if i == 3:
                        ax.legend(loc = [0., -0.5], ncol = 1)
                    else:
                        ax.legend(loc = [0, -0.5], ncol = 2)
                else:
                    if i == 3:
                        ax.legend(loc = [0., -0.4], ncol = 1)
                    else:
                        ax.legend(loc = [0, -0.4], ncol = 2)

            ax.axhline(5.5, ls = '--', lw = 0.5)
            ax.axhline(10.5, ls = '--', lw = 0.5)
            ax.axhline(15.5, ls = '--', lw = 0.5)
            ax.axhline(20.5, ls = '--', lw = 0.5)

            print(matplotlib.get_backend())
        for ax, lab in zip(np.ravel(axes), ascii_lowercase):
            ax.text(0, 1.05, lab, transform=ax.transAxes, fontweight = 'bold')

    fig.text(0.5, 0.496, f'Sobel\'s main effect sensitivity indices of {name_list[p]}', ha='center', fontsize = 12)
    fig.text(0.5, 0.075, f'Sobel\'s total effect sensitivity indices of {name_list[p]}', ha='center', fontsize = 12)
    fig.text(0.05, 0.5, 'Modeled C-balance variables', va='center', rotation='vertical', fontsize = 12)

    fig.savefig(os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 
                             f'plot_ensemble_UQ_{prefix}.png'), dpi = 600.,
                bbox_inches = 'tight')

module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend

In [3]:
fig, axes = plt.subplots(3, 4, figsize = (16, 11), sharex = True, sharey = True)
fig.subplots_adjust(wspace = 0.0, hspace = 0.4)
for p, prefix in enumerate(prefix_list):
    f = open(os.path.join(os.environ['HOME'], 'models', 'OLMT', 'pklfiles', 
             f'{prefix}_US-SPR_ICB20TRCNPRDCTCBC.pkl'), 'rb')
    mycase = pickle.load(f)
    f.close()

    #Total sensitivity indices
    for i, (pft,pftname) in enumerate(zip([2, 3, 11, 0], pft_names[:-1] + ['Column'])):
        subset = np.where(np.array(mycase.ensemble_pfts) == pft)[0]

        ax = axes[p, i]

        bottom = np.zeros(len(x_pos))
        for s in subset:
            temp = np.array([mycase.sens_tot[v][s,0] for v in variable_list])
            temp = temp[reorder]
            if p == 0:
                ax.bar(x_pos, temp, align='center', # alpha=0.5,
                    bottom = bottom, label = mycase.ensemble_parms[s])
            else:
                ax.bar(x_pos, temp, align='center', # alpha=0.5,
                    bottom = bottom, label = parmfile_to_paper[mycase.ensemble_parms[s]])
            bottom = bottom + temp

        # add a line for total
        total = np.array([mycase.sens_tot[v][:,0].sum() for v in variable_list])
        total = total[reorder]
        ax.plot(x_pos, total, '-k', label = 'Sum of all\nperturbed parameters')

        ax.set_xlim([x_pos[0]-0.5, x_pos[-1]+0.5])
        ax.set_xticks(x_pos)
        ax.set_xticklabels([ticklabels[t] for t in reorder], rotation=90)
        ax.tick_params(axis='x', pad=10)

        if p == 2:
            for pp, pname in enumerate(['Column'] + pft_names):
                ax.text(0.1 + pp*0.2, -0.075, pname, transform = ax.transAxes, 
                        ha = 'center', color = '#1f77b4')

        if p == 0:
            ax.set_title(f'{pftname} parameters')

        ax.axvline(5.5, ls = '--', lw = 0.5)
        ax.axvline(10.5, ls = '--', lw = 0.5)
        ax.axvline(15.5, ls = '--', lw = 0.5)
        ax.axvline(20.5, ls = '--', lw = 0.5)

        if i == 0:
            if p == 2:
                ax.legend(loc = [0.2, -0.7], ncol = 5)
            else:
                ax.legend(loc = [0.2, -0.25], ncol = 5)

        if i == 3:
            if p == 0:
                ax.legend(loc = [-0.24, -0.25], ncol = 2)
            elif p == 1:
                ax.legend(loc = [0, -0.25], ncol = 2)
            else:
                ax.legend(loc = [0, -0.7], ncol = 2)

        if i == 0:
            ax.set_ylabel(name_list[p])
    for ax, lab in zip(np.ravel(axes), ascii_lowercase):
        ax.text(-0.05, 1.05, lab, transform=ax.transAxes, fontweight = 'bold')
fig.savefig(os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 
                         'plot_ensemble_UQ_tot.png'), dpi = 600.,
            bbox_inches = 'tight')
plt.show()

<Figure size 1600x1200 with 8 Axes>

<Figure size 1600x1200 with 8 Axes>

<Figure size 1600x1200 with 8 Axes>

<Figure size 1600x1100 with 12 Axes>

In [4]:
fig, axes = plt.subplots(2, 2, figsize = (16, 18))

for p, prefix in enumerate(prefix_list):
    f = open(os.path.join(os.environ['HOME'], 'models', 'OLMT', 'pklfiles', 
             f'{prefix}_US-SPR_ICB20TRCNPRDCTCBC.pkl'), 'rb')
    mycase = pickle.load(f)
    f.close()

    varlist_reorder = [variable_list[reorder[i]] for i in range(len(variable_list))]
    ticks_reorder = [ticklabels[reorder[i]] for i in range(len(ticklabels))]

    allcorr = pd.DataFrame(np.nan, index = varlist_reorder, columns = varlist_reorder)
    all_p = pd.DataFrame(np.nan, index = varlist_reorder, columns = varlist_reorder)

    for i in range(len(variable_list)):
        for j in range(i):
            v1 = variable_list[reorder[i]]
            v2 = variable_list[reorder[j]]

            x1 = mycase.output[v1][0,:]
            x2 = mycase.output[v2][0,:]
            filt = ~(np.isnan(x1) | np.isnan(x2))

            rho, pval = pearsonr(x1[filt], x2[filt])

            allcorr.loc[v1, v2] = rho
            allcorr.loc[v2, v1] = rho
            all_p.loc[v1, v2] = pval
            all_p.loc[v2, v1] = pval

    ax = axes.flat[p]
    cf = ax.imshow(allcorr, vmin = 0, vmax = 1, cmap = 'Reds')

    mask = all_p.values <= 0.05           # Boolean array, same shape as all_p
    for (row, col), sig in np.ndenumerate(mask):
        if sig:
            ax.text(col,                       # x‑coordinate (imshow uses col,row order)
                    row,                       # y‑coordinate
                    'x',
                    ha='center', va='center',
                    color='black', fontsize=10, weight='bold')
        ax.set_title(name_list[p])
        ax.set_xticks(range(len(ticks_reorder)))
        ax.set_yticks(range(len(ticks_reorder)))

    ax.set_yticklabels(ticks_reorder)
    ax.set_xticklabels(ticks_reorder, rotation = 90)

    ax.axhline(4.5, ls = '--', lw = 0.5)
    ax.axhline(9.5, ls = '--', lw = 0.5)
    ax.axhline(14.5, ls = '--', lw = 0.5)
    ax.axhline(19.5, ls = '--', lw = 0.5)

    ax.axvline(4.5, ls = '--', lw = 0.5)
    ax.axvline(9.5, ls = '--', lw = 0.5)
    ax.axvline(14.5, ls = '--', lw = 0.5)
    ax.axvline(19.5, ls = '--', lw = 0.5)

ax = axes.flat[-1]
ax.axis('off')

cax = fig.add_axes([0.6, 0.05, 0.01, 0.4])
plt.colorbar(cf, cax = cax, orientation = 'vertical')

for ax, lab in zip(np.ravel(axes)[:-1], ascii_lowercase):
    ax.text(0, 1.05, lab, transform=ax.transAxes, fontweight = 'bold')

fig.savefig(os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 
                         'plot_ensemble_UQ_correlation.png'), dpi = 600.,
            bbox_inches = 'tight')